In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "models").is_dir() and (path / "datasets").is_dir() and (path / "utils").is_dir():
            return path
    raise RuntimeError("Repository root was not found from the current working directory.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) in sys.path:
    sys.path.remove(str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)


In [1]:
from utils.visualization import show_result
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import csv
import os


import config_mv
from models.patchcore import PatchCore
from models.backbone import get_backbone
from datasets.mvtec import MyData
from utils.metrics import get_image_auc
from utils.visualization import show_result, denormalize

In [2]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(device)

mps


In [3]:
classn = ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut'
          ,'leather','metal_nut','pill','screw','tile','toothbrush','transistor'
          ,'wood','zipper']

file_path = "results.csv"
file_exists = os.path.isfile(file_path)

results = []  # ⭐ 결과 먼저 저장

for name in classn:
    train_data = MyData(
        name,
        phase="train",
        batch_size=config_mv.BATCH_SIZE,
        shuffle=False,
        limit=config_mv.TRAIN_LIMIT,
    )

    test_data = MyData(
        name,
        phase="test",
        batch_size=config_mv.BATCH_SIZE,
        shuffle=False,
        limit=config_mv.TEST_LIMIT,
        limit_per_class=config_mv.TEST_LIMIT_PER_CLASS,
    )

    backbone = get_backbone()
    patchcore = PatchCore(backbone, k=config_mv.K, device=device)

    patchcore.fit(train_data)

    scores = []
    for i in range(len(test_data)):
        img, label = test_data[i]
        score, _ = patchcore.predict(img)
        scores.append(score.item())

    # top score
    top_score = max(scores)

    # AUC
    image_auc = get_image_auc(test_data, patchcore)

    # ⭐ 결과 저장 (소수점 처리 포함)
    results.append([
        name,
        config_mv.TRAIN_LIMIT,
        config_mv.K,
        round(image_auc, 2),
        round(top_score, 2)
    ])

# 🔥 AUC 기준 내림차순 정렬
results = sorted(results, key=lambda x: x[3], reverse=True)

# 🔥 CSV 저장 (한 번만)
file_path = "results.csv"
with open(file_path, 'w', newline="") as f:  # 'w'로 덮어쓰기
    writer = csv.writer(f)

    writer.writerow(["category","train_limit","k","auc","top_score"])  # header 한 번만
    writer.writerows(results)

In [4]:
import pandas as pd

df = pd.read_csv("results.csv")
df

,category,train_limit,k,auc,top_score
0,bottle,50,50,1.00,6.54
1,leather,50,50,1.00,8.58
2,wood,50,50,1.00,9.51
3,zipper,50,50,0.98,7.43
4,hazelnut,50,50,0.97,9.49
5,pill,50,50,0.92,7.22
6,toothbrush,50,50,0.89,6.42
7,capsule,50,50,0.87,7.08
8,cable,50,50,0.86,7.84
9,carpet,50,50,0.84,5.20
